# Spain Blackout 28 Apr 2025 — Dataset EDA
Quick exploration of the three training sheets before running the PINN.

In [ ]:
import sys
sys.path.insert(0, '/mnt/data/hackathon/venv_pkgs')
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from data_loader import load_all, load_hourly, load_pre_event_15min, load_second_by_second

EXCEL = '../data/Spain_Blackout_28Apr2025_Dataset.xlsx'
bundle = load_all(EXCEL)
print('Loaded OK')

## 1 · Hourly Demand & Generation Mix

In [ ]:
df_h = bundle.df_hourly
print(df_h[['hour','demand','solar','wind','hydro','nuclear','ccgt','total_gen','renewable_frac']].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Spain 28 Apr 2025 — Hourly Grid State', fontsize=13)

# Stacked generation
ax = axes[0]
sources = ['solar', 'wind', 'hydro', 'nuclear', 'ccgt', 'other']
colors  = ['#fdae61', '#2c7bb6', '#1a9641', '#d7191c', '#756bb1', '#969696']
bottom = np.zeros(len(df_h))
hours  = range(len(df_h))
for src, col in zip(sources, colors):
    ax.bar(hours, df_h[src], bottom=bottom, color=col, label=src.capitalize(), alpha=0.85)
    bottom += df_h[src].values
ax.plot(hours, df_h['demand'], 'k-o', linewidth=2, markersize=4, label='Demand')
ax.set_xlabel('Hour (CEST)')
ax.set_ylabel('MW')
ax.set_title('Generation Mix vs Demand')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Renewable fraction
ax = axes[1]
ax.plot(hours, df_h['renewable_frac'] * 100, color='#2c7bb6', linewidth=2)
ax.fill_between(hours, df_h['renewable_frac'] * 100, alpha=0.15, color='#2c7bb6')
ax.axhline(70, color='orange', linestyle='--', label='70% threshold')
ax.set_xlabel('Hour (CEST)')
ax.set_ylabel('Renewable fraction (%)')
ax.set_title('Renewable Penetration — risk builds above 70%')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/eda_hourly.png', dpi=130)
plt.show()

## 2 · Pre-Event 15-min Resolution — Inertia Decline

In [ ]:
df_15 = bundle.df_15min
print(df_15[['time','frequency','H_est','renewable_frac','notes']].to_string())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
fig.suptitle('Spain 28 Apr 2025 — 15-min Pre-Event Data', fontsize=13)

t_min = df_15['t_s'] / 60

axes[0].plot(t_min, df_15['frequency'], 'b-o', linewidth=2)
axes[0].axhline(49.98, color='orange', linestyle='--', label='49.98 Hz (first oscillation)')
axes[0].set_ylabel('Frequency (Hz)')
axes[0].set_title('Frequency')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(t_min, df_15['H_est'], 'r-o', linewidth=2, label='Estimated H')
axes[1].axhline(1.14, color='k', linestyle='--', label='Spain event H = 1.14 s')
axes[1].set_ylabel('Inertia H (s)')
axes[1].set_title('System Inertia Constant — declining as renewables dominate')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

axes[2].plot(t_min, df_15['renewable_frac'] * 100, 'g-o', linewidth=2)
axes[2].set_ylabel('Renewable % ')
axes[2].set_xlabel('Minutes after 11:00 CEST')
axes[2].set_title('Renewable Penetration')
axes[2].grid(alpha=0.3)

for ax in axes:
    ax.axvline(63, color='purple', linestyle=':', alpha=0.6)   # 12:03 first oscillation
    ax.axvline(79, color='purple', linestyle=':', alpha=0.6)   # 12:19 second oscillation

plt.tight_layout()
plt.savefig('../outputs/eda_15min.png', dpi=130)
plt.show()

## 3 · Second-by-Second Collapse

In [ ]:
df_1s = bundle.df_1s
print(df_1s[['t_s','frequency','dfdt','H','delta_P','event_marker']]
      .dropna(subset=['event_marker']).to_string())

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=True)
fig.suptitle('Spain 28 Apr 2025 — Second-by-Second Collapse (12:32–12:33:30 CEST)', fontsize=13)

t = df_1s['t_s']

axes[0].plot(t, df_1s['frequency'], 'b-', linewidth=2)
axes[0].axhline(49.8, color='orange', linestyle='--', linewidth=1, label='49.8 Hz relay')
axes[0].axhline(47.5, color='red',    linestyle=':',  linewidth=1, label='47.5 Hz UFLS')
axes[0].set_ylabel('Frequency (Hz)')
axes[0].set_title('Frequency Collapse')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(t, df_1s['dfdt'], 'r-', linewidth=2)
axes[1].axhline(0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_ylabel('df/dt (Hz/s)')
axes[1].set_title('Rate of Change of Frequency (RoCoF)')
axes[1].grid(alpha=0.3)

axes[2].plot(t, df_1s['H'], 'g-', linewidth=2)
axes[2].axhline(1.14, color='k', linestyle='--', label='H=1.14s pre-event')
axes[2].set_ylabel('Inertia H (s)')
axes[2].set_title('Inertia Collapse During Cascade')
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

axes[3].plot(t, df_1s['delta_P'], 'k-', linewidth=2)
axes[3].axhline(0, color='k', linestyle='-', linewidth=0.5)
axes[3].set_ylabel('ΔP = Pm − Pe (MW)')
axes[3].set_xlabel('Seconds from 12:32:00 CEST')
axes[3].set_title('Power Imbalance')
axes[3].grid(alpha=0.3)

# Annotate key events
events = df_1s.dropna(subset=['event_marker'])
for _, row in events.iterrows():
    for ax in axes:
        ax.axvline(row['t_s'], color='purple', linestyle=':', alpha=0.5)
    axes[0].text(row['t_s'] + 0.3, 1.0,
                 row['event_marker'][:30], fontsize=6,
                 rotation=90, va='bottom', color='purple')

plt.tight_layout()
plt.savefig('../outputs/eda_1s.png', dpi=130)
plt.show()

## 4 · Normalised Tensors (PINN Input Preview)

In [ ]:
print('Phase-1 normalised tensors (first 5 rows):')
import torch
for name, tensor in [
    ('t15',   bundle.t15_norm),
    ('Pm15',  bundle.pm15_norm),
    ('Pe15',  bundle.pe15_norm),
    ('rf15',  bundle.rf15_norm),
    ('f15',   bundle.f15_norm),
]:
    print(f'  {name:8s}: {tensor[:5].numpy().round(4)}')

print('\nPhase-2 normalised tensors (first 5 rows):')
for name, tensor in [
    ('t1s',   bundle.t1s_norm),
    ('Pm1s',  bundle.pm1s_norm),
    ('Pe1s',  bundle.pe1s_norm),
    ('rf1s',  bundle.rf1s_norm),
    ('f1s',   bundle.f1s_norm),
]:
    print(f'  {name:8s}: {tensor[:5].numpy().round(4)}')

print('\nScalers (min, max):')
for k, (lo, hi) in bundle.scalers.items():
    print(f'  {k:6s}: [{lo:.2f}, {hi:.2f}]')